# Session1_Task8_short — Product Performance & Price Optimization

In [ ]:
import pandas as pd, numpy as np, warnings; warnings.filterwarnings('ignore')

s = pd.read_csv('sales_transactions_cleaned.csv')
p = pd.read_csv('products.csv')
s['revenue'] = (s['quantity']*s['price']) - pd.to_numeric(s['discount_amount'],errors='coerce').fillna(0)
s['date'] = pd.to_datetime(s['date'])
s['month'] = s['date'].dt.to_period('M').astype(str)

# clean price/cost
clean = lambda col: pd.to_numeric(col.astype(str).str.replace(r'[^0-9.\-]','',regex=True),errors='coerce').abs()
p['cost_clean'] = clean(p['cost'])

# product performance: qty, revenue, profit_margin
perf = (s.groupby('product_id').agg(total_quantity_sold=('quantity','sum'), total_revenue=('revenue','sum'))
         .reset_index()
         .merge(p[['product_id','cost_clean']], on='product_id', how='left')
         .assign(total_cost=lambda x: x['total_quantity_sold']*x['cost_clean'],
                 profit_margin=lambda x: ((x['total_revenue']-x['total_quantity_sold']*x['cost_clean'])/x['total_revenue']).round(4))
        [['product_id','total_quantity_sold','total_revenue','profit_margin']]
         .sort_values('total_revenue', ascending=False).round(2))

# price elasticity: PED = % change qty / % change price
m = (s.groupby(['product_id','month']).agg(qty=('quantity','sum'),avg_price=('price','mean'))
      .reset_index().sort_values(['product_id','month']))
m['ped'] = (m.groupby('product_id')['qty'].pct_change() /
             m.groupby('product_id')['avg_price'].pct_change()).replace([np.inf,-np.inf],np.nan)
ped = (m.groupby('product_id')['ped'].mean().reset_index().round(4)
        .rename(columns={'ped':'price_elasticity_of_demand'})
        .assign(suggested_price_change=lambda x: x['price_elasticity_of_demand'].apply(
            lambda v: '-5%' if abs(v)>1 else '+5%' if pd.notna(v) else '0%')))

perf.to_csv('Session5_Product_Performance_short_short.csv', index=False)
ped.to_csv('Session5_Price_Analysis_short.csv', index=False)
print('✅ Saved Session5_Product_Performance_short.csv & Session5_Price_Analysis_short.csv')
# จุดสังเกต: profit_margin อยู่ 0-1, suggested_price_change เป็น +5% หรือ -5%